# 11 · Key reconciliation — making Alice's and Bob's keys match

After sifting, Alice's and Bob's keys **still disagree on the error bits** (about QBER of
them). A raw QKD run therefore produces two *almost*-identical keys — useless as a shared
secret until they're reconciled. This notebook tests the **Cascade** reconciliation now
wired into the distributed BB84 path.

What Cascade does: Bob corrects his key toward Alice's by exchanging block *parities* over
the classical link (`qne/cascade.py`, driven via `qne-sequence/reconcile_link.py`). Every
parity revealed leaks a little to an eavesdropper — that `bits_leaked` is tracked and
subtracted in the secure-key accounting. Above the ~11% QBER threshold the run **aborts**
instead of reconciling.

**Runs locally over loopback — no FABRIC slice.** It launches the real two-process
`node_runner` (Bob + Alice on `127.0.0.1`), the same code path that runs on the testbed.

## 1 · Setup — run a two-process BB84 session

In [ ]:
import sys, os, json, subprocess, time, math
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PKG = PROJECT_DIR / 'qne-sequence'
_PORT = [57720]

def run_bb84(fidelity=0.95, reconcile=True, eve_fraction=0.0, num_pulses=8000,
             sample_fraction=0.2, seed=7):
    """Launch Bob + Alice on loopback; return (alice_result, bob_result)."""
    _PORT[0] += 2
    port = _PORT[0]
    env = dict(os.environ)
    env['PYTHONPATH'] = f"{PKG}{os.pathsep}{PROJECT_DIR}"
    def spawn(role, peer):
        return subprocess.Popen(
            [sys.executable, '-m', 'qne_sequence.node_runner',
             '--role', role, '--name', role, '--peer', peer,
             '--host', '127.0.0.1', '--port', str(port),
             '--num-pulses', str(num_pulses), '--fidelity', str(fidelity),
             '--sample-fraction', str(sample_fraction), '--seed', str(seed),
             '--eve-fraction', str(eve_fraction),
             '--reconcile' if reconcile else '--no-reconcile'],
            cwd=str(PKG), env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    bob = spawn('bob', 'alice'); time.sleep(1); alice = spawn('alice', 'bob')
    a_out, a_err = alice.communicate(timeout=120)
    b_out, _ = bob.communicate(timeout=30)
    def parse(out, err=''):
        line = next((ln for ln in out.splitlines() if ln.startswith('{')), '')
        assert line, f"no result:\n{err}"
        return json.loads(line)
    return parse(a_out, a_err), parse(b_out)

def mismatch(a, b):
    """Fraction of key bits where Alice and Bob differ."""
    if not a['key'] or not b['key']:
        return None
    return bin(a['key'] ^ b['key']).count('1') / max(a['key_bits'], 1)

# smoke check
a, b = run_bb84(fidelity=0.95, reconcile=True)
print(f"ran: qber={a['qber']:.4f}  key_bits={a['key_bits']}  reconciled={a['reconciled']}")

## 2 · With vs without reconciliation

Same noisy channel (F = 0.95, so QBER ≈ 2.5%), run both ways. Without reconciliation the
keys differ on ~QBER of the bits; with it, they should be **identical**.

In [ ]:
raw_a, raw_b = run_bb84(fidelity=0.95, reconcile=False)
rec_a, rec_b = run_bb84(fidelity=0.95, reconcile=True)

print(f"{'':22}{'QBER':>8}{'key_bits':>10}{'keys match':>12}{'mismatch':>10}"
      f"{'corrections':>13}{'leaked':>9}{'secure_bits':>13}")
for label, a, b in [("no reconciliation", raw_a, raw_b), ("with Cascade", rec_a, rec_b)]:
    print(f"{label:22}{a['qber']:>8.4f}{a['key_bits']:>10}"
          f"{str(a['key'] == b['key']):>12}{(mismatch(a, b) or 0):>10.4f}"
          f"{str(a['corrections']):>13}{str(a['bits_leaked']):>9}{str(a['secure_key_bits']):>13}")

## 3 · What reconciliation costs

Cascade isn't free — correcting more errors means revealing more parities. Sweep the
channel fidelity (hence QBER) and look at the **leakage** and the **Cascade efficiency**
`f = bits_leaked / (key_bits · H(QBER))` — the ratio to the Shannon limit; ~1.1–1.5 for moderate QBER, and notably
higher at very low QBER where the fixed per-block parity overhead dominates. More leakage means fewer secret bits survive.

In [ ]:
import matplotlib.pyplot as plt

def H(x):
    return 0.0 if x <= 0 or x >= 1 else -x*math.log2(x) - (1-x)*math.log2(1-x)

rows = []
for F in [0.99, 0.97, 0.95, 0.93, 0.90]:
    a, b = run_bb84(fidelity=F, reconcile=True)
    q = a['qber']; kb = a['key_bits']; leaked = a['bits_leaked']
    eff = leaked / (kb * H(q)) if kb and H(q) > 0 else float('nan')
    rows.append({'F': F, 'qber': q, 'leak_rate': leaked / kb, 'eff': eff,
                 'secure_bits': a['secure_key_bits'], 'matched': a['key'] == b['key']})
    print(f"F={F}  QBER={q:.4f}  leaked/bit={leaked/kb:.3f}  efficiency f={eff:.2f}  "
          f"secure_bits={a['secure_key_bits']}  keys_match={a['key']==b['key']}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
qs = [r['qber']*100 for r in rows]
ax1.plot(qs, [r['leak_rate'] for r in rows], 'o-', label='leaked bits / key bit')
ax1.plot(qs, [r['eff'] for r in rows], 's-', color='#F06FA6', label='Cascade efficiency f')
ax1.set(xlabel='QBER (%)', ylabel='ratio', title='Reconciliation cost'); ax1.grid(alpha=.3); ax1.legend(fontsize=9)
ax2.plot(qs, [r['secure_bits'] for r in rows], 'o-', color='#43D0E3')
ax2.set(xlabel='QBER (%)', ylabel='secure key bits', title='Secret bits left after reconciliation + PA'); ax2.grid(alpha=.3)
fig.tight_layout()
out = PROJECT_DIR / 'paper' / 'figures' / 'reconciliation.png'
out.parent.mkdir(parents=True, exist_ok=True); fig.savefig(out, dpi=150, bbox_inches='tight')
print('saved', out); plt.show()

## 4 · Eavesdropper → abort above the threshold

Reconciliation only runs when a secure key is still possible (QBER < ~11%). Turn on a full
intercept-resend eavesdropper (`eve_fraction=1.0`, QBER → ~25%): the run should **refuse to
reconcile** — `reconciled=False`, keys stay different, `secure_key_bits=0`.

In [ ]:
atk_a, atk_b = run_bb84(fidelity=1.0, reconcile=True, eve_fraction=1.0)
print(f"under attack: qber={atk_a['qber']:.4f}  reconciled={atk_a['reconciled']}  "
      f"keys_match={atk_a['key']==atk_b['key']}  secure_fraction={atk_a['secure_fraction']}  "
      f"secure_key_bits={atk_a['secure_key_bits']}")

## 5 · Verify

In [ ]:
checks = []
def rec(name, ok, detail=''):
    checks.append(ok); print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f' — {detail}' if detail else ''))

rec('noisy channel had real errors', raw_a['qber'] > 0, f"QBER={raw_a['qber']:.4f}")
rec('without reconciliation keys DIFFER', raw_a['key'] != raw_b['key'])
rec('mismatch tracks QBER (not ~50%)', 0 < (mismatch(raw_a, raw_b) or 1) < 0.1)
rec('with Cascade keys MATCH bit-for-bit', rec_a['key'] == rec_b['key'])
rec('Cascade actually did work', rec_a['corrections'] > 0 and rec_a['bits_leaked'] > 0)
rec('reconciled run has positive secure_key_bits', rec_a['secure_key_bits'] > 0)
rec('efficiency f in a sane range (1.0–3.0)', all(1.0 <= r['eff'] <= 3.0 for r in rows))
rec('eavesdropped run ABORTED (no reconcile)', not atk_a['reconciled'] and atk_a['secure_key_bits'] == 0)

print('\nALL CHECKS PASSED' if checks and all(checks) else '\nSOME CHECKS FAILED')

## Takeaways

- A raw sifted key is **two almost-matching keys** — reconciliation is what turns them into
  one shared secret.
- Cascade makes them **identical bit-for-bit**, at the cost of leaking parities
  (`bits_leaked`), which privacy amplification then removes — so the secret shrinks with
  the error rate.
- Security is preserved by **aborting above ~11% QBER** rather than reconciling a key that
  can't be made secret.

**Related:** `--no-reconcile` (used here) is exactly the raw-key mode the eavesdropper
detectability project studies (notebook 10 + the assignment); this notebook shows what the
*full* protocol does with those keys afterward.